# Train all consortium experts on Kaggle

This notebook creates `e_math`, `e_code`, and `e_instr` with SFT, then `e_pref` with DPO, all from one shared base. It reuses the repository data-preparation and training scripts.

Before running: enable Internet and select a GPU. The notebook clones this public repository directly from GitHub. The default `configs/model_config.yaml` targets Qwen2.5-1.5B-Instruct with fp16 for Kaggle T4. Edit that file in the repo before pushing changes to GitHub. Run cells in order. Completed adapters are skipped and interrupted SFT runs resume from an epoch checkpoint.

This notebook trains the initial experts only; it does not run merges or E4/E5 workflows.

In [1]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

GPU 0: Tesla T4 (UUID: GPU-5b76d201-e22e-684e-6c3c-29ca1482e39b)
GPU 1: Tesla T4 (UUID: GPU-235be3a3-c54c-0e34-fa45-299805084ee4)
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
!pip install -q "transformers<5" "trl<0.20" "peft" "accelerate"
!pip install -q -U "torchao>=0.16.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/abhinavpareek655/LLM_Merging.git"
REPO_DIR = Path("/kaggle/working/LLM_Merging")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} exists but is not a Git checkout; remove it or choose another REPO_DIR.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / "src" / "train" / "train_sft.py").is_file():
    raise FileNotFoundError(f"Expected training code was not found after cloning {REPO_URL}.")
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)

Cloning into '/kaggle/working/LLM_Merging'...


Repository: /kaggle/working/LLM_Merging


Updating files: 100% (287/287), done.


In [4]:
# Install training dependencies without replacing Kaggle's PyTorch build.
%pip install -q -U "transformers>=4.48.0" "peft>=0.14.0" "trl>=0.14.0" "datasets>=3.0.0" "accelerate>=1.2.0" "pyyaml>=6.0"

import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.11.0+cu130 CUDA: True


In [5]:
import json
import yaml

with open("configs/model_config.yaml", encoding="utf-8") as f:
    MODEL_CONFIG = yaml.safe_load(f)
OUTPUT_ROOT = Path(MODEL_CONFIG["output_root"])
if not OUTPUT_ROOT.is_absolute():
    OUTPUT_ROOT = REPO_DIR / OUTPUT_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for key, value in {
    "LLM_MODEL_ID": MODEL_CONFIG["model_id"],
    "LLM_TORCH_DTYPE": MODEL_CONFIG.get("torch_dtype", "float16"),
    "LLM_ATTN_IMPLEMENTATION": MODEL_CONFIG.get("attention_implementation", "sdpa"),
    "LLM_MAX_SEQ_LENGTH": str(MODEL_CONFIG.get("max_seq_length", 1024)),
    "LLM_LORA_R": str(MODEL_CONFIG.get("lora", {}).get("rank", 16)),
    "LLM_LORA_ALPHA": str(MODEL_CONFIG.get("lora", {}).get("alpha", 32)),
    "LLM_LORA_DROPOUT": str(MODEL_CONFIG.get("lora", {}).get("dropout", 0.05)),
    "LLM_LORA_BIAS": str(MODEL_CONFIG.get("lora", {}).get("bias", "none")),
    "LLM_OUTPUT_ROOT": str(OUTPUT_ROOT),
}.items():
    os.environ[key] = value

LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = OUTPUT_ROOT / "outputs" / "datasets"
ADAPTER_DIR = OUTPUT_ROOT / "outputs" / "adapters"
print("Model:", MODEL_CONFIG["model_id"])
print("Precision:", MODEL_CONFIG.get("torch_dtype", "float16"))
print("Output:", OUTPUT_ROOT)
print("Experts:", ", ".join(MODEL_CONFIG["experts"]))

Model: Qwen/Qwen2.5-1.5B-Instruct
Precision: float16
Output: /kaggle/working/qwen2.5-1.5b-experts
Experts: math, code, instruction, preference


In [6]:
import subprocess
import sys

def run_logged(name, command):
    log_path = LOG_DIR / f"{name}.log"
    print("Running:", " ".join(map(str, command)))
    with log_path.open("a", encoding="utf-8") as log:
        result = subprocess.run(list(map(str, command)), cwd=REPO_DIR, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-60:]))
        raise subprocess.CalledProcessError(result.returncode, command)
    print(f"Completed {name}; log: {log_path}")

run_logged("prepare_data", [sys.executable, "src/data/prepare_data.py"])
expected = {"e_math.jsonl": 5000, "e_code.jsonl": 3000, "e_instr.jsonl": 3000, "e_pref_train.jsonl": 2000, "e_pref_test.jsonl": 500}
for filename, minimum in expected.items():
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Expected dataset missing: {path}")
    with path.open(encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())
    print(f"{filename}: {count}")
    if count < minimum:
        raise RuntimeError(f"{filename}: expected at least {minimum} rows, found {count}.")

Running: /usr/bin/python3 src/data/prepare_data.py
Completed prepare_data; log: /kaggle/working/qwen2.5-1.5b-experts/logs/prepare_data.log
e_math.jsonl: 5000
e_code.jsonl: 3000
e_instr.jsonl: 3000
e_pref_train.jsonl: 2000
e_pref_test.jsonl: 500


In [7]:
# SFT domain experts, one after another to avoid GPU contention.
for expert in ("math", "code", "instr"):
    adapter_path = ADAPTER_DIR / f"e_{expert}"
    adapter_file = adapter_path / "adapter_model.safetensors"
    if adapter_file.is_file():
        print(f"Skipping completed e_{expert}: {adapter_file}")
        continue
    run_logged(f"train_sft_{expert}", [
        sys.executable, "src/train/train_sft.py",
        "--dataset_path", DATA_DIR / f"e_{expert}.jsonl",
        "--output_dir", adapter_path,
        "--config_path", "configs/sft_kaggle_config.yaml",
    ])
    if not adapter_file.is_file():
        raise FileNotFoundError(f"SFT adapter was not saved: {adapter_file}")

Running: /usr/bin/python3 src/train/train_sft.py --dataset_path /kaggle/working/qwen2.5-1.5b-experts/outputs/datasets/e_math.jsonl --output_dir /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_math --config_path configs/sft_kaggle_config.yaml
Completed train_sft_math; log: /kaggle/working/qwen2.5-1.5b-experts/logs/train_sft_math.log
Running: /usr/bin/python3 src/train/train_sft.py --dataset_path /kaggle/working/qwen2.5-1.5b-experts/outputs/datasets/e_code.jsonl --output_dir /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_code --config_path configs/sft_kaggle_config.yaml
Completed train_sft_code; log: /kaggle/working/qwen2.5-1.5b-experts/logs/train_sft_code.log
Running: /usr/bin/python3 src/train/train_sft.py --dataset_path /kaggle/working/qwen2.5-1.5b-experts/outputs/datasets/e_instr.jsonl --output_dir /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_instr --config_path configs/sft_kaggle_config.yaml
Completed train_sft_instr; log: /kaggle/working/qwen2.5-1.5b-exp

In [8]:
# Precompute reference log probabilities from the shared base, then train the DPO preference expert.
REF_DATA = DATA_DIR / "e_pref_train_with_ref.jsonl"
run_logged("precompute_pref_reference", [sys.executable, "src/data/precompute_ref_logps.py"])
if not REF_DATA.is_file():
    raise FileNotFoundError(f"DPO reference dataset was not created: {REF_DATA}")

pref_adapter = ADAPTER_DIR / "e_pref"
pref_file = pref_adapter / "adapter_model.safetensors"
if pref_file.is_file():
    print("Skipping completed e_pref:", pref_file)
else:
    run_logged("train_dpo_pref", [
        sys.executable, "src/train/train_dpo_stable.py",
        "--dataset_path", REF_DATA,
        "--output_dir", pref_adapter,
        "--config_path", "configs/dpo_config.yaml",
    ])
if not pref_file.is_file():
    raise FileNotFoundError(f"DPO adapter was not saved: {pref_file}")

Running: /usr/bin/python3 src/data/precompute_ref_logps.py
Completed precompute_pref_reference; log: /kaggle/working/qwen2.5-1.5b-experts/logs/precompute_pref_reference.log
Running: /usr/bin/python3 src/train/train_dpo_stable.py --dataset_path /kaggle/working/qwen2.5-1.5b-experts/outputs/datasets/e_pref_train_with_ref.jsonl --output_dir /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_pref --config_path configs/dpo_config.yaml
Completed train_dpo_pref; log: /kaggle/working/qwen2.5-1.5b-experts/logs/train_dpo_pref.log


In [9]:
import zipfile

manifest = {
    "model_id": MODEL_CONFIG["model_id"],
    "torch_dtype": MODEL_CONFIG.get("torch_dtype", "float16"),
    "max_seq_length": MODEL_CONFIG.get("max_seq_length", 1024),
    "lora": MODEL_CONFIG.get("lora", {}),
    "experts": {},
}
for expert in ("math", "code", "instr", "pref"):
    adapter_path = ADAPTER_DIR / f"e_{expert}"
    adapter_file = adapter_path / "adapter_model.safetensors"
    if not adapter_file.is_file():
        raise FileNotFoundError(f"Expert adapter is missing: {adapter_file}")
    manifest["experts"][f"e_{expert}"] = str(adapter_path)

manifest_path = OUTPUT_ROOT / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive_path = OUTPUT_ROOT / "consortium_experts.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path, manifest_path.relative_to(OUTPUT_ROOT))
    for folder in (ADAPTER_DIR, DATA_DIR):
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_ROOT))
print("All four adapters are ready:")
for name, path in manifest["experts"].items():
    print(f"  {name}: {path}")
print("Archive:", archive_path)
print("Use Kaggle's Save Version / notebook output to persist or download it.")

All four adapters are ready:
  e_math: /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_math
  e_code: /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_code
  e_instr: /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_instr
  e_pref: /kaggle/working/qwen2.5-1.5b-experts/outputs/adapters/e_pref
Archive: /kaggle/working/qwen2.5-1.5b-experts/consortium_experts.zip
Use Kaggle's Save Version / notebook output to persist or download it.
